# Cervello-Bambino — training su Colab

Notebook per eseguire le fasi di training intensive (PyTorch, `cervello/`) su GPU Colab invece che sul PC locale (solo CPU).

Prima di lanciare le celle: **Runtime -> Cambia tipo di runtime -> GPU**.

## 1. Verifica GPU

In [ ]:
!nvidia-smi

## 2. Clona il repository

In [ ]:
REPO_URL = "https://github.com/Mecks3D/llm-test.git"

# Torna sempre a /content prima di clonare: rieseguire questa cella
# mentre si e' gia' dentro llm-test clonerebbe il repo dentro se stesso.
%cd /content
!rm -rf llm-test
!git clone $REPO_URL
%cd llm-test


## 3. Installa le dipendenze

In [ ]:
!pip install -q -r requirements.txt

## 4. Verifica che torch veda la GPU

In [ ]:
import torch
print("CUDA disponibile:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 5. Monta Google Drive (per salvare i checkpoint)

La sessione Colab perde tutto alla scadenza: i checkpoint vanno salvati su Drive, non nel filesystem locale del runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/cervello-bambino/checkpoint"
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoint dir:", CHECKPOINT_DIR)

## 6. Smoke test: la suite di test passa nell'ambiente Colab?

Prima di lanciare training lunghi, verifica che l'ambiente sia equivalente a quello locale.

In [ ]:
!python -m pytest -q

## 6b. Canary di apprendimento (Fase 2a, gruppo 6)

Test "cancello duro": 32 esempi di stadio 1, overfit fino a esattezza-risposta 100% (max 1000 step). Marcato `slow` (escluso dalla suite di default): su CPU e' impraticabile, va eseguito qui con GPU.

In [ ]:
!python -m pytest -q -m slow -s


## 6c. Run di fumo (Fase 2a, tappa T6)

Prima del training vero: genera il dataset ridotto (200 storie, solo stadio 1) e allena per 300 step con `configs/v1_fumo.yaml`. Verifica che la loss scenda e che compaiano log/checkpoint in `dati/risultati/v1_fumo/`. Le prime valutazioni possono essere lente: il modello appena inizializzato non ha ancora imparato a emettere `[FINE]`, quindi la decodifica greedy può arrivare fino al tetto `ctx=3072` prima di fermarsi. Se una singola valutazione impiega più di qualche minuto, interrompi e segnalalo.

In [ ]:
!python -m esami.genera --config configs/v1_fumo.yaml --stadio 1


In [ ]:
!python -m cervello.addestra --config configs/v1_fumo.yaml --stadio 1


In [ ]:
import json, shutil, os

with open('dati/risultati/v1_fumo/log.jsonl', encoding='utf-8') as f:
    righe = [json.loads(r) for r in f]
for r in righe:
    print(r)

# Esito dell'esame (esattezza per tipo + conteggi di calibrazione:
# 'errore' = struttura giusta ma contenuto sbagliato -> poco training;
# 'malformata' dominante -> problema di decodifica, indagare).
with open('dati/risultati/v1_fumo/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Il filesystem di Colab e' effimero: copia i risultati su Drive.
dest = os.path.join(CHECKPOINT_DIR, 'v1_fumo')
shutil.copytree('dati/risultati/v1_fumo', dest, dirs_exist_ok=True)
print('risultati copiati su Drive ->', dest)


## 6d. Run di fumo LUNGO (verifica post-revisione T1-T6)

Stadio 1 con 1000 storie e max 3000 step (`configs/v1_fumo_lungo.yaml`, ~20-30 min).
Obiettivo: vedere `esattezza_dev` **salire sopra lo zero** (conferma che la catena
training→decodifica→grafo impara anche fuori dall'overfit del canary) e misurare
quanti step servono per avvicinare la soglia 0.95 — l'ancora per stimare la durata
del curriculum completo (decisione 10). Se raggiunge soglia+0.01 per 2 valutazioni
consecutive si ferma da solo prima dei 3000 step.


In [ ]:
!python -m esami.genera --config configs/v1_fumo_lungo.yaml --stadio 1


In [ ]:
!python -m cervello.addestra --config configs/v1_fumo_lungo.yaml --stadio 1


In [ ]:
import json, shutil, os

with open('dati/risultati/v1_fumo_lungo/log.jsonl', encoding='utf-8') as f:
    righe = [json.loads(r) for r in f]
for r in righe:
    print(r)

# Esito dell'esame, con i primi campioni non esatti (generato vs oro):
# le 'malformate' si guardano, non si indovinano.
with open('dati/risultati/v1_fumo_lungo/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Il filesystem di Colab e' effimero: copia i risultati su Drive.
dest = os.path.join(CHECKPOINT_DIR, 'v1_fumo_lungo')
shutil.copytree('dati/risultati/v1_fumo_lungo', dest, dirs_exist_ok=True)
print('risultati copiati su Drive ->', dest)


## 6e. Prova del checkpoint intra-stadio su Drive (da fare PRIMA di T7)

Prova end-to-end, tutta automatica (~5 minuti): lancia il run di fumo con
`--copia-sicurezza`, lo **uccide** (SIGKILL, come un runtime che muore) appena il
primo checkpoint parziale compare su Drive, cancella la cartella locale dei
risultati (come farebbe un runtime nuovo), poi rilancia lo stesso comando e
verifica che riprenda da Drive e arrivi in fondo. Eseguire le 5 celle in ordine;
l'ultima stampa `PROVA SUPERATA` se tutto funziona.


In [ ]:
# 6e-1: cartella di prova su Drive (pulita) e niente residui locali.
import os, shutil
COPIA_DIR_FUMO = os.path.join(CHECKPOINT_DIR, 'v1_fumo_prova')
shutil.rmtree(COPIA_DIR_FUMO, ignore_errors=True)
os.makedirs(COPIA_DIR_FUMO, exist_ok=True)
shutil.rmtree('dati/risultati/v1_fumo', ignore_errors=True)
print('cartella di prova su Drive:', COPIA_DIR_FUMO)


In [ ]:
# 6e-2: dataset di fumo (idempotente, veloce).
!python -m esami.genera --config configs/v1_fumo.yaml --stadio 1


In [ ]:
# 6e-3: run interrotto. Il training parte in un sottoprocesso e viene UCCISO
# appena stadio1_parziale.pt compare su Drive (prima valutazione, step 100,
# ~1-2 min). Poi si cancella la cartella locale: resta SOLO la copia su Drive.
import os, shutil, subprocess, time

# Si aspettano ENTRAMBI i file (parziale + log): vengono copiati in
# quest'ordine e uccidere in mezzo lascerebbe il log indietro di un giro.
parziale_drive = os.path.join(COPIA_DIR_FUMO, 'stadio1_parziale.pt')
log_drive = os.path.join(COPIA_DIR_FUMO, 'log.jsonl')
with open('fumo_interrotto.log', 'w') as log:
    proc = subprocess.Popen(
        ['python', '-m', 'cervello.addestra', '--config', 'configs/v1_fumo.yaml',
         '--stadio', '1', '--copia-sicurezza', COPIA_DIR_FUMO],
        stdout=log, stderr=subprocess.STDOUT)
    attesa = 0
    while proc.poll() is None and not (os.path.exists(parziale_drive) and os.path.exists(log_drive)):
        time.sleep(5)
        attesa += 5
        if attesa % 30 == 0:
            print(f'... in attesa del primo parziale su Drive ({attesa}s)')
    if proc.poll() is not None:
        print(open('fumo_interrotto.log').read())
        raise RuntimeError('il training e\' finito senza mai scrivere il parziale su Drive')
    proc.kill()
    proc.wait()

print(open('fumo_interrotto.log').read())
print('>>> training UCCISO dopo il primo parziale su Drive:', parziale_drive)
shutil.rmtree('dati/risultati/v1_fumo', ignore_errors=True)
print('>>> cartella locale dei risultati cancellata (simula un runtime nuovo)')


In [ ]:
# 6e-4: ripresa. Stesso identico comando: deve stampare 'recuperato dalla
# copia di sicurezza' e 'ripresa intra-stadio da ... step 100', poi finire
# i 300 step e l'esame (che a 300 step FALLISCE: atteso, non e' la prova).
!python -m cervello.addestra --config configs/v1_fumo.yaml --stadio 1 --copia-sicurezza "{COPIA_DIR_FUMO}"


In [ ]:
# 6e-5: verifica finale.
import os, json

for nome in ('stadio1.pt', 'esame_stadio1.json', 'log.jsonl'):
    p = os.path.join(COPIA_DIR_FUMO, nome)
    assert os.path.exists(p) and os.path.getsize(p) > 0, f'manca su Drive: {p}'
assert not os.path.exists(os.path.join(COPIA_DIR_FUMO, 'stadio1_parziale.pt')), \
    'il parziale doveva sparire a stadio completato'

with open('dati/risultati/v1_fumo/log.jsonl', encoding='utf-8') as f:
    steps = [json.loads(r)['step'] for r in f]
print('step nel log (curva intera, senza buchi):', steps)
assert steps == [100, 200, 300], steps

print()
print('PROVA SUPERATA: parziale su Drive, ripresa automatica, log intero,')
print('pulizia a fine stadio. Si puo\' procedere con la sezione 7 (T7).')


## 7. T7 — Stadio 1 di produzione (config v1)

Primo stadio del run vero: 5000 storie corte, max 20.000 step, eval ogni 500
(`configs/v1.yaml`). **Stima: ~2,5-3h nel caso peggiore** (meno se l'esattezza dev
raggiunge soglia+0.01 per 2 valutazioni consecutive e si ferma da solo) — run > 2h,
va lanciato solo con decisione esplicita (FASE2_PIANO.md, decisione 10).
Al termine esegue l'esame di stadio 1 (soglia 0.95) e salva checkpoint + esito.
Gli stadi 2-3 si lanciano DOPO, con `--stadio 2` / `--stadio 3` (riprendono dal
checkpoint precedente), una volta ristimata la durata dai dati di questo run.

**Checkpoint intra-stadio e ripresa**: a ogni valutazione (500 step) il training
salva `stadio1_parziale.pt` in locale e, grazie a `--copia-sicurezza`, lo replica
subito su Drive (scrittura locale + copia: scrivere direttamente sul mount di
Drive dentro il training loop è notoriamente inaffidabile). Se il runtime muore:
nuovo runtime, celle 1-5 e clone, poi rilanciare le stesse celle qui sotto —
il training recupera da solo il parziale da Drive e riprende (messaggio
`ripresa intra-stadio da ...`). Perdita massima: 500 step (~10-15 min).
Per ripartire invece DA ZERO, cancellare prima `stadio1_parziale.pt` dalla
cartella su Drive.


In [ ]:
# Checkpoint, log ed esiti vengono replicati su Drive dal training stesso
# (--copia-sicurezza, cella successiva): qui si controlla solo se su Drive
# c'e' gia' un parziale da cui il run riprenderebbe.
import os
COPIA_DIR = os.path.join(CHECKPOINT_DIR, 'v1')
os.makedirs(COPIA_DIR, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR)
parziale = os.path.join(COPIA_DIR, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')


In [ ]:
!python -m esami.genera --config configs/v1.yaml --stadio 1


In [ ]:
!python -m cervello.addestra --config configs/v1.yaml --stadio 1 --copia-sicurezza "{COPIA_DIR}"


In [ ]:
import json

with open('dati/risultati/v1/log.jsonl', encoding='utf-8') as f:
    righe = [json.loads(r) for r in f]
for r in righe:
    print(r)

with open('dati/risultati/v1/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Niente copia manuale: checkpoint, log ed esame sono gia' stati replicati
# su Drive dal training (--copia-sicurezza).


## 8. Curriculum completo in un comando solo (riferimento)

`python -m cervello.addestra --config configs/v1.yaml` esegue stadi 1-3 con esami
ai cancelli (il criterio di accettazione "run riproducibile da un comando solo").
Stima nel caso peggiore ~20h: su Colab conviene il percorso per-stadio della
sezione 7. Prima di lanciare gli stadi 2-3 servono anche i loro dataset:
`!python -m esami.genera --config configs/v1.yaml --stadio 2` (e `--stadio 3`).


In [ ]:
!python -m cervello.addestra --config configs/v1.yaml


## 9. Stadio "facile" (cast ridotto) — riparte dal checkpoint di v1

Curriculum separato (`configs/v1_facile.yaml`, run `v1_facile`): storie con
solo 3 personaggi (anna, piero, maria) invece di 6, per ridurre la
confusione di binding entità→luogo osservata nell'esame dello stadio 1 di
v1 (0.573 di esattezza, soglia 0.95 — vedi analisi del log). Non riparte da
pesi casuali: usa `--pesi-iniziali` per caricare i pesi già addestrati di
`v1/stadio1.pt` (già su Drive, replicato dalla sezione 7).

Va lanciato DOPO la sezione 7 (serve `v1/stadio1.pt` su Drive). Dataset e
step molto più leggeri del run di produzione (1500 storie, max 6000 step).

**Stessa protezione da interruzioni della sezione 7**: `--copia-sicurezza`
replica su Drive checkpoint intra-stadio, log ed esito a ogni valutazione;
se il runtime muore, rilanciare le stesse celle qui sotto e il training
riprende da solo dal parziale (messaggio `ripresa intra-stadio da ...`).
Per ripartire da zero, cancellare prima `stadio1_parziale.pt` dalla
cartella `v1_facile` su Drive.

In [ ]:
# Stessa logica della sezione 7: cartella di copia di sicurezza dedicata a
# questa run, piu' il checkpoint di partenza (stadio 1 di v1, cast pieno)
# gia' su Drive.
import os
COPIA_DIR_FACILE = os.path.join(CHECKPOINT_DIR, 'v1_facile')
os.makedirs(COPIA_DIR_FACILE, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR_FACILE)

PESI_INIZIALI = os.path.join(CHECKPOINT_DIR, 'v1', 'stadio1.pt')
assert os.path.exists(PESI_INIZIALI), (
    f"manca il checkpoint di partenza: {PESI_INIZIALI} "
    "(eseguire prima la sezione 7)"
)
print('pesi iniziali <-', PESI_INIZIALI)

parziale = os.path.join(COPIA_DIR_FACILE, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')

In [ ]:
!python -m esami.genera --config configs/v1_facile.yaml --stadio 1

In [ ]:
!python -m cervello.addestra --config configs/v1_facile.yaml --stadio 1 --pesi-iniziali "{PESI_INIZIALI}" --copia-sicurezza "{COPIA_DIR_FACILE}"

In [ ]:
import json

with open('dati/risultati/v1_facile/log.jsonl', encoding='utf-8') as f:
    righe = [json.loads(r) for r in f]
for r in righe:
    print(r)

with open('dati/risultati/v1_facile/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Niente copia manuale: checkpoint, log ed esame sono gia' stati replicati
# su Drive dal training (--copia-sicurezza).

## 10. Esperimento anti-scorciatoia (v1_anti)

Run separata (`configs/v1_anti.yaml`, run `v1_anti`): stesso cast PIENO e
stessa distribuzione dev/esame di `v1` (confronto diretto con lo 0,573
dell'esame di stadio 1), ma il train usa due leve sui dati (fasi/
FASE2_PIANO_ANTISCORCIATOIA.md): selezione anti-scorciatoia (sovracampiona
le domande "difficili", dove la scorciatoia frequenza/recency sbaglia) e
storie troncate a tick intermedi (supervisione densa del tracking). Da
**pesi casuali** (niente `--pesi-iniziali`): l'attribuzione del risultato
dev'essere pulita. Budget alleggerito su richiesta di Andrea: 6.000 storie,
max 8.000 step, **stima ~1-1,2h**.

**Prima il fumo** (`configs/v1_anti_fumo.yaml`, ~10 min): verifica che la
pipeline intera (selezione + troncamenti + best-dev) funzioni, e che la
composizione del train stampata sia sensata (quota difficili ≈0,6, quota
non-lo-so ≈15-20%) prima di lanciare il run vero. Il run vero (10b) parte
SOLO dopo l'ok esplicito di Andrea al fumo (stessa regola delle sezioni
precedenti — decisione 10 di FASE2_PIANO.md).

In [ ]:
# Copia di sicurezza dedicata al fumo (stessa logica delle sezioni 7/9).
import os
COPIA_DIR_ANTI_FUMO = os.path.join(CHECKPOINT_DIR, 'v1_anti_fumo')
os.makedirs(COPIA_DIR_ANTI_FUMO, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR_ANTI_FUMO)
parziale = os.path.join(COPIA_DIR_ANTI_FUMO, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')


In [ ]:
!python -m esami.genera --config configs/v1_anti_fumo.yaml --stadio 1


In [ ]:
# Composizione del train (verifica visiva della quota anti-scorciatoia,
# piano §5.4): conteggio difficile/facile/non-lo-so e record pieni/troncati.
import json
from collections import Counter

def stampa_composizione(percorso_train):
    conteggio_record = Counter()
    conteggio_difficolta = Counter()
    n_esempi = 0
    with open(percorso_train, encoding='utf-8') as f:
        for riga in f:
            r = json.loads(riga)
            conteggio_record['troncato' if 'troncamento' in r else 'pieno'] += 1
            for es in r['esempi']:
                n_esempi += 1
                conteggio_difficolta[es.get('difficolta', 'n/d')] += 1
    print('record:', dict(conteggio_record))
    print('esempi:', n_esempi)
    for k, v in conteggio_difficolta.items():
        print(f'  {k}: {v} ({v / n_esempi:.3f})')

stampa_composizione('dati/anti_fumo/stadio1/train.jsonl')


In [ ]:
!python -m cervello.addestra --config configs/v1_anti_fumo.yaml --stadio 1 --copia-sicurezza "{COPIA_DIR_ANTI_FUMO}"


In [ ]:
# Cancello del fumo (piano §7, T5): pipeline OK, composizione sensata (sopra),
# esattezza_dev DEVE salire sopra zero.
import json

with open('dati/risultati/v1_anti_fumo/log.jsonl', encoding='utf-8') as f:
    for r in [json.loads(r) for r in f]:
        print(r)

with open('dati/risultati/v1_anti_fumo/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))


### 10b. Run vero (v1_anti) — SOLO dopo l'ok esplicito di Andrea al fumo

Stima ~1-1,2h. Al termine: esame ufficiale di stadio 1 (confrontabile
direttamente con lo 0,573 di `v1`) e `esami.diagnosi` sia su `stadio1.pt`
(ultimo checkpoint) sia su `stadio1_best.pt` (miglior `esattezza_dev`
durante il run — può differire dall'ultimo, vedi T3): la diagnosi rende
permanenti le metriche dell'analisi qualitativa (baseline euristiche,
esattezza condizionata, anatomia degli errori) sui criteri di lettura del
piano (§8).

In [ ]:
# Copia di sicurezza dedicata al run vero.
import os
COPIA_DIR_ANTI = os.path.join(CHECKPOINT_DIR, 'v1_anti')
os.makedirs(COPIA_DIR_ANTI, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR_ANTI)
parziale = os.path.join(COPIA_DIR_ANTI, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')


In [ ]:
!python -m esami.genera --config configs/v1_anti.yaml --stadio 1


In [ ]:
stampa_composizione('dati/anti/stadio1/train.jsonl')


In [ ]:
!python -m cervello.addestra --config configs/v1_anti.yaml --stadio 1 --copia-sicurezza "{COPIA_DIR_ANTI}"


In [ ]:
import json

with open('dati/risultati/v1_anti/log.jsonl', encoding='utf-8') as f:
    for r in [json.loads(r) for r in f]:
        print(r)

with open('dati/risultati/v1_anti/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Niente copia manuale: checkpoint, log ed esame sono gia' stati replicati
# su Drive dal training (--copia-sicurezza).


In [ ]:
# Diagnosi (piano §5.3) su ENTRAMBI i checkpoint: l'ultimo e il best-dev
# (possono differire, vedi T3). diagnosi.py non ha --copia-sicurezza: si
# copiano i JSON su Drive a mano qui sotto.
import shutil

!python -m esami.diagnosi --config configs/v1_anti.yaml --stadio 1 --checkpoint dati/risultati/v1_anti/stadio1.pt --split esame
shutil.copy2('dati/risultati/v1_anti/diagnosi_stadio1.json', os.path.join(COPIA_DIR_ANTI, 'diagnosi_stadio1.json'))

!python -m esami.diagnosi --config configs/v1_anti.yaml --stadio 1 --checkpoint dati/risultati/v1_anti/stadio1_best.pt --split esame
shutil.copy2('dati/risultati/v1_anti/diagnosi_stadio1.json', os.path.join(COPIA_DIR_ANTI, 'diagnosi_stadio1_best.json'))

print('diagnosi copiate su', COPIA_DIR_ANTI)


## 11. Curriculum a cast crescente (1 -> 2 -> 3 personaggi)

Tre run separate (`configs/v1_grad1.yaml`, `v1_grad2.yaml`, `v1_grad3.yaml`), da lanciare IN ORDINE, ognuna a partire dal checkpoint best-dev della precedente (`--pesi-iniziali`). Idea di Andrea: invece del salto secco a 3 personaggi di v1_facile.yaml (sezione 9), salire di difficolta' un personaggio alla volta, con storie molto corte (1-3 tick, non 3-6) per isolare la variabile "numero di entita' in gioco". Budget di step crescente a ogni gradino (1500 -> 2500 -> 4000).

Punto di partenza dell'INTERO curriculum: il best-dev di `v1_anti` (sezione 10b, cast pieno, meno bias di scorciatoia) — deve gia' essere su Drive in `CHECKPOINT_DIR/v1_anti/stadio1_best.pt` prima di lanciare 11a.

**Nota sulla composizione dei dati** (verificata in locale prima di scrivere questi config): con 1 solo personaggio e 1-3 tick la quota di "non lo so" e' molto piu' alta del solito (~53% invece di ~15-20%, perche' quasi nulla viene stabilito in cosi' poco tempo) e ci sono meno esempi per storia (~3,8 invece di 8). Scende a ~37% (gradino 2) e ~29% (gradino 3) man mano che il cast cresce. Non e' un bug: il gradino 1 insegna soprattutto grammatica e calibrazione dell'astensione, non ancora il tracking vero e proprio — da tenere d'occhio nei risultati, non necessariamente un problema.

**Stessa protezione da interruzioni delle sezioni precedenti**: `--copia-sicurezza` replica su Drive checkpoint intra-stadio, log ed esito a ogni valutazione; se il runtime muore, rilanciare le stesse celle e il training riprende da solo dal parziale.

### 11a. Gradino 1/3 — 1 personaggio (anna)

Riparte dal best-dev di `v1_anti` (sezione 10b). Dataset e step piu' leggeri (2000 storie, max 1500 step).

**`training.early_stop: false`** (aggiunto su richiesta di Andrea): il run precedente si era fermato da solo a step ~800 appena `esattezza_dev` superava soglia+0,01 per 2 valutazioni consecutive. Ora `v1_grad1.yaml` disattiva quel criterio: il training arriva sempre a max_step=1500, indipendentemente da quanto presto supera la soglia (le altre run — v1, v1_anti, v1_facile, grad2/3 — restano col comportamento di sempre, invariato).

In [ ]:
# Da eseguire SOLO se si vuole rifare il gradino 1 da zero (come qui:
# il run precedente e' gia' completo). Senza questo cleanup, log.jsonl
# verrebbe recuperato dalla copia su Drive e il nuovo run ACCODEREBBE
# le sue righe a quelle del run vecchio (comportamento voluto per una
# ripresa dopo interruzione, non per un restart pulito).
import shutil

for cartella in (
    'dati/risultati/v1_grad1',
    os.path.join(CHECKPOINT_DIR, 'v1_grad1'),
):
    if os.path.exists(cartella):
        shutil.rmtree(cartella)
        print('rimossa:', cartella)


In [ ]:
# Copia di sicurezza dedicata a questo gradino, piu' il checkpoint
# di partenza (best-dev del gradino precedente).
import os
COPIA_DIR_V1_GRAD1 = os.path.join(CHECKPOINT_DIR, 'v1_grad1')
os.makedirs(COPIA_DIR_V1_GRAD1, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR_V1_GRAD1)

PESI_INIZIALI = os.path.join(CHECKPOINT_DIR, 'v1_anti', 'stadio1_best.pt')
assert os.path.exists(PESI_INIZIALI), (
    f"manca il checkpoint di partenza: {PESI_INIZIALI}"
)
print('pesi iniziali <-', PESI_INIZIALI)

parziale = os.path.join(COPIA_DIR_V1_GRAD1, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')


In [ ]:
!python -m esami.genera --config configs/v1_grad1.yaml --stadio 1

In [ ]:
!python -m cervello.addestra --config configs/v1_grad1.yaml --stadio 1 --pesi-iniziali "{PESI_INIZIALI}" --copia-sicurezza "{COPIA_DIR_V1_GRAD1}"

In [ ]:
import json

with open('dati/risultati/v1_grad1/log.jsonl', encoding='utf-8') as f:
    for r in [json.loads(r) for r in f]:
        print(r)

with open('dati/risultati/v1_grad1/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Niente copia manuale: checkpoint, log ed esame sono gia' stati
# replicati su Drive dal training (--copia-sicurezza).


### 11b. Gradino 2/3 — 2 personaggi (anna, piero)

Va lanciato DOPO 11a (serve `v1_grad1/stadio1_best.pt` su Drive). 2500 storie, max 2500 step.

In [ ]:
# Copia di sicurezza dedicata a questo gradino, piu' il checkpoint
# di partenza (best-dev del gradino precedente).
import os
COPIA_DIR_V1_GRAD2 = os.path.join(CHECKPOINT_DIR, 'v1_grad2')
os.makedirs(COPIA_DIR_V1_GRAD2, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR_V1_GRAD2)

PESI_INIZIALI = os.path.join(CHECKPOINT_DIR, 'v1_grad1', 'stadio1_best.pt')
assert os.path.exists(PESI_INIZIALI), (
    f"manca il checkpoint di partenza: {PESI_INIZIALI}"
)
print('pesi iniziali <-', PESI_INIZIALI)

parziale = os.path.join(COPIA_DIR_V1_GRAD2, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')


In [ ]:
!python -m esami.genera --config configs/v1_grad2.yaml --stadio 1

In [ ]:
!python -m cervello.addestra --config configs/v1_grad2.yaml --stadio 1 --pesi-iniziali "{PESI_INIZIALI}" --copia-sicurezza "{COPIA_DIR_V1_GRAD2}"

In [ ]:
import json

with open('dati/risultati/v1_grad2/log.jsonl', encoding='utf-8') as f:
    for r in [json.loads(r) for r in f]:
        print(r)

with open('dati/risultati/v1_grad2/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Niente copia manuale: checkpoint, log ed esame sono gia' stati
# replicati su Drive dal training (--copia-sicurezza).


### 11c. Gradino 3/3 — 3 personaggi (anna, piero, maria)

Va lanciato DOPO 11b (serve `v1_grad2/stadio1_best.pt` su Drive). 3000 storie, max 4000 step. Stesso cast finale di `v1_facile.yaml` (sezione 9): a fine run confrontare l'esame con 0,7448 per vedere se la gradualita' ha aiutato a parita' di personaggi.

In [ ]:
# Copia di sicurezza dedicata a questo gradino, piu' il checkpoint
# di partenza (best-dev del gradino precedente).
import os
COPIA_DIR_V1_GRAD3 = os.path.join(CHECKPOINT_DIR, 'v1_grad3')
os.makedirs(COPIA_DIR_V1_GRAD3, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR_V1_GRAD3)

PESI_INIZIALI = os.path.join(CHECKPOINT_DIR, 'v1_grad2', 'stadio1_best.pt')
assert os.path.exists(PESI_INIZIALI), (
    f"manca il checkpoint di partenza: {PESI_INIZIALI}"
)
print('pesi iniziali <-', PESI_INIZIALI)

parziale = os.path.join(COPIA_DIR_V1_GRAD3, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')


In [ ]:
!python -m esami.genera --config configs/v1_grad3.yaml --stadio 1

In [ ]:
!python -m cervello.addestra --config configs/v1_grad3.yaml --stadio 1 --pesi-iniziali "{PESI_INIZIALI}" --copia-sicurezza "{COPIA_DIR_V1_GRAD3}"

In [ ]:
import json

with open('dati/risultati/v1_grad3/log.jsonl', encoding='utf-8') as f:
    for r in [json.loads(r) for r in f]:
        print(r)

with open('dati/risultati/v1_grad3/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Niente copia manuale: checkpoint, log ed esame sono gia' stati
# replicati su Drive dal training (--copia-sicurezza).


## 12. Esperimento "tempo" (v1_tempo)

Linea parallela alla roadmap A2/B/C (fasi/FASE2_PIANO_TEMPO.md), decisa con
Andrea il 2026-07-11: un solo personaggio per storia (rotante tra i 6,
`dataset.cast_rotante: true`), storie lunghe 12-24 tick, domande NUOVE
condizionate nel tempo/luogo (`posizione_tempo`, `azione_tempo`,
`azione_luogo`) accanto a `posizione`. Misura una capacita' a monte del
binding multi-entita': seguire come cambia lo stato di UNA persona nel
tempo — se riesce, il muro multi-entita' e' interferenza (rafforza Fase B/C2
del piano di diagnosi); se fallisce anche qui, il tracking temporale in se'
non si forma a questa scala/budget (rafforza A2/C1).

Da **pesi casuali** (niente `--pesi-iniziali`): attribuzione pulita, i
checkpoint esistenti porterebbero dentro la scorciatoia frequenza/recency
diagnosticata in FASE2_PIANO_DIAGNOSI.md. Dati propri (`dati/tempo/`): il
curriculum ufficiale e i config esistenti non cambiano. Sequenze composte
molto piu' corte di `v1_anti` a parita' di step (misurato in locale: max
~514 token, ctx=3072, ampio margine) -> **stima ~1-1,5h** per 3.000 storie,
8.000 step.

**Prima il fumo** (`configs/v1_tempo_fumo.yaml`, 300 storie, 300 step):
verifica che la pipeline intera (cast rotante + domande tempo + best-dev)
funzioni e che la composizione del train sia sensata, prima di lanciare il
run vero. Il run vero (12b) parte SOLO dopo l'ok esplicito di Andrea al
fumo (stessa regola delle sezioni precedenti — decisione 10 di
FASE2_PIANO.md).

In [ ]:
# Copia di sicurezza dedicata al fumo (stessa logica delle sezioni 7/9/10).
import os
COPIA_DIR_TEMPO_FUMO = os.path.join(CHECKPOINT_DIR, 'v1_tempo_fumo')
os.makedirs(COPIA_DIR_TEMPO_FUMO, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR_TEMPO_FUMO)
parziale = os.path.join(COPIA_DIR_TEMPO_FUMO, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')


In [ ]:
!python -m esami.genera --config configs/v1_tempo_fumo.yaml --stadio 1


In [ ]:
# Composizione del train "tempo" (cancello del fumo, piano fasi/FASE2_PIANO_TEMPO.md
# §5/T5): a differenza di stampa_composizione (sezione 10, pensata per la quota
# difficile/facile/non-lo-so di v1_anti), qui l'asse rilevante è il tipo — quattro
# tipi (posizione, posizione_tempo, azione_tempo, azione_luogo) e la quota non-lo-so
# per tipo (~0 per i tre tipi nuovi per costruzione, concentrata su "posizione" —
# vedi nota epistemica in mondo/domande.py).
import json
from collections import Counter

def stampa_composizione_tempo(percorso_train):
    conteggio_record = Counter()
    conteggio_tipo = Counter()
    non_lo_so_per_tipo = Counter()
    n_esempi = 0
    with open(percorso_train, encoding='utf-8') as f:
        for riga in f:
            r = json.loads(riga)
            conteggio_record['troncato' if 'troncamento' in r else 'pieno'] += 1
            for es in r['esempi']:
                n_esempi += 1
                conteggio_tipo[es['tipo']] += 1
                if 'non-lo-so' in es['risposta']:
                    non_lo_so_per_tipo[es['tipo']] += 1
    print('record:', dict(conteggio_record))
    print('esempi:', n_esempi)
    for tipo, n in conteggio_tipo.items():
        quota_nls = non_lo_so_per_tipo[tipo] / n
        print(f'  {tipo}: {n} ({n / n_esempi:.3f})  non-lo-so: {non_lo_so_per_tipo[tipo]} ({quota_nls:.3f})')

stampa_composizione_tempo('dati/tempo_fumo/stadio1/train.jsonl')


In [ ]:
!python -m cervello.addestra --config configs/v1_tempo_fumo.yaml --stadio 1 --copia-sicurezza "{COPIA_DIR_TEMPO_FUMO}"


In [ ]:
# Cancello del fumo (piano §6, T5): pipeline OK, composizione sensata (sopra),
# esattezza_dev DEVE salire sopra zero.
import json

with open('dati/risultati/v1_tempo_fumo/log.jsonl', encoding='utf-8') as f:
    for r in [json.loads(r) for r in f]:
        print(r)

with open('dati/risultati/v1_tempo_fumo/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))


### 12b. Run vero (v1_tempo) — SOLO dopo l'ok esplicito di Andrea al fumo

Stima ~1-1,5h. Al termine: esame ufficiale di stadio 1 (`esattezza_per_tipo`
per leggere `posizione_tempo`/`azione_tempo`/`azione_luogo` separatamente),
lo split diagnostico `tracking_tempo.jsonl` (analogo ad A3: solo domande
`posizione_tempo` dove l'oro non e' ne' la scorciatoia di frequenza ne' lo
stato finale ne' roba fresca) e `esami.diagnosi` su ENTRAMBI i checkpoint
(l'ultimo e il best-dev) per gli split `esame` e `tracking-tempo`. Criteri
di lettura fissati PRIMA del run in fasi/FASE2_PIANO_TEMPO.md §7.

In [ ]:
# Copia di sicurezza dedicata al run vero.
import os
COPIA_DIR_TEMPO = os.path.join(CHECKPOINT_DIR, 'v1_tempo')
os.makedirs(COPIA_DIR_TEMPO, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR_TEMPO)
parziale = os.path.join(COPIA_DIR_TEMPO, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')


In [ ]:
!python -m esami.genera --config configs/v1_tempo.yaml --stadio 1


In [ ]:
import json
from collections import Counter

def stampa_composizione_tempo(percorso_train):
    conteggio_record = Counter()
    conteggio_tipo = Counter()
    non_lo_so_per_tipo = Counter()
    n_esempi = 0
    with open(percorso_train, encoding='utf-8') as f:
        for riga in f:
            r = json.loads(riga)
            conteggio_record['troncato' if 'troncamento' in r else 'pieno'] += 1
            for es in r['esempi']:
                n_esempi += 1
                conteggio_tipo[es['tipo']] += 1
                if 'non-lo-so' in es['risposta']:
                    non_lo_so_per_tipo[es['tipo']] += 1
    print('record:', dict(conteggio_record))
    print('esempi:', n_esempi)
    for tipo, n in conteggio_tipo.items():
        quota_nls = non_lo_so_per_tipo[tipo] / n
        print(f'  {tipo}: {n} ({n / n_esempi:.3f})  non-lo-so: {non_lo_so_per_tipo[tipo]} ({quota_nls:.3f})')

stampa_composizione_tempo('dati/tempo/stadio1/train.jsonl')


In [ ]:
!python -m cervello.addestra --config configs/v1_tempo.yaml --stadio 1 --copia-sicurezza "{COPIA_DIR_TEMPO}"


In [ ]:
import json

with open('dati/risultati/v1_tempo/log.jsonl', encoding='utf-8') as f:
    for r in [json.loads(r) for r in f]:
        print(r)

with open('dati/risultati/v1_tempo/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Niente copia manuale: checkpoint, log ed esame sono gia' stati replicati
# su Drive dal training (--copia-sicurezza).


In [ ]:
# Split diagnostico tracking_tempo.jsonl (analogo ad A3, piano §4.3): non ha
# --copia-sicurezza (e' generato da esami.genera, non dal training), si
# rigenera qui e basta (deterministico, stessi seed dell'esame ufficiale).
!python -m esami.genera --config configs/v1_tempo.yaml --stadio 1 --split tracking-tempo


In [ ]:
# Diagnosi (piano §4.3/§7) su ENTRAMBI i checkpoint, split esame e
# tracking-tempo. diagnosi.py non ha --copia-sicurezza: si copiano i JSON
# su Drive a mano qui sotto (stessa logica della sezione 10, cella 53).
import shutil

for checkpoint, suffisso in (('stadio1.pt', ''), ('stadio1_best.pt', '_best')):
    for split in ('esame', 'tracking-tempo'):
        comando = (
            f'python -m esami.diagnosi --config configs/v1_tempo.yaml --stadio 1 '
            f'--checkpoint dati/risultati/v1_tempo/{checkpoint} --split {split}'
        )
        get_ipython().system(comando)
        nome_split = '' if split == 'esame' else f'_{split}'
        sorgente = f'dati/risultati/v1_tempo/diagnosi_stadio1{nome_split}.json'
        dest = os.path.join(COPIA_DIR_TEMPO, f'diagnosi_stadio1{nome_split}{suffisso}.json')
        shutil.copy2(sorgente, dest)

print('diagnosi copiate su', COPIA_DIR_TEMPO)


## 13. Fase B — supervisione densa in-sequenza dello stato (v1_stato)

Fase B della roadmap di diagnosi (fasi/FASE2_PIANO_STATO.md): dentro la stessa
sequenza, a ogni fine tick il modello emette il grafo dello stato corrente
(dove si trova ciascuna persona), poi la storia prosegue; la domanda resta
DOPO la storia. Costringe a *mantenere* lo stato di tutte le entità lungo il
contesto (attacca l'interferenza multi-entità) e l'etichetta di tick esplicita
dà un indice temporale discreto (attacca il "puntatore temporale sfocato"
isolato in fasi/FASE2_PIANO_TEMPO.md §8).

Cast pieno, da **pesi casuali** (niente `--pesi-iniziali`: attribuzione
pulita), dati propri (`dati/stato/`). dev/esame ufficiali INVARIATI, senza
blocchi stato → confronto diretto con **v1: 0,573**. Solo il train ha i blocchi
`[STATO]`; all'esame li genera il modello (decode interlacciato, §5).
Lunghezze reali misurate in locale: max ~1231 token, ctx=3072, ampio margine.

**Prima il fumo** (`configs/v1_stato_fumo.yaml`, 300 storie, 600 step):
verifica che la pipeline intera giri (dati con blocchi stato, maschera stato,
esame interlacciato) e che la composizione sia sensata. Il run vero (13b)
parte SOLO dopo l'ok esplicito di Andrea al fumo.


In [ ]:
# Copia di sicurezza dedicata al fumo (stessa logica delle sezioni 7/10/12).
import os
COPIA_DIR_STATO_FUMO = os.path.join(CHECKPOINT_DIR, 'v1_stato_fumo')
os.makedirs(COPIA_DIR_STATO_FUMO, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR_STATO_FUMO)
parziale = os.path.join(COPIA_DIR_STATO_FUMO, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')


In [ ]:
!python -m esami.genera --config configs/v1_stato_fumo.yaml --stadio 1


In [ ]:
# Composizione del train "stato" (cancello del fumo, fasi/FASE2_PIANO_STATO.md §6.3/T5):
# proporzione di token-stato vs eventi, numero di blocchi [STATO], e le lunghezze
# reali delle sequenze composte contro ctx (il cancello dichiarato in T5).
import json
from cervello.sequenza import componi_esempio, span_blocchi_stato

def stampa_composizione_stato(percorso_train, ctx=3072):
    n_record = n_esempi = n_blocchi = tot_tok = tok_stato = 0
    lung = []
    with open(percorso_train, encoding='utf-8') as f:
        for riga in f:
            r = json.loads(riga); n_record += 1
            storia = r['storia']
            spans = span_blocchi_stato(storia)
            n_blocchi += len(spans)
            tok_stato += sum(fine - inizio for inizio, fine in spans)
            tot_tok += len(storia)
            for es in r['esempi']:
                n_esempi += 1
                lung.append(len(componi_esempio([storia], es['domanda'], es['risposta'])))
    lung.sort()
    print(f'record: {n_record}, esempi: {n_esempi}, blocchi [STATO]: {n_blocchi}')
    print(f'token storia: {tot_tok}, di cui stato: {tok_stato} ({tok_stato / max(tot_tok,1):.1%})')
    print(f'lunghezza composta: mediana={lung[len(lung)//2]} '
          f'p99={lung[int(len(lung)*0.99)]} MAX={lung[-1]} (ctx={ctx})')
    print(f'esempi che sforano ctx: {sum(1 for x in lung if x > ctx)}')

stampa_composizione_stato('dati/stato_fumo/stadio1/train.jsonl')


In [ ]:
!python -m cervello.addestra --config configs/v1_stato_fumo.yaml --stadio 1 --copia-sicurezza "{COPIA_DIR_STATO_FUMO}"


In [ ]:
# Cancello del fumo (T5): pipeline OK, composizione sensata (sopra),
# esattezza_dev (decode interlacciato) DEVE salire sopra zero.
import json

with open('dati/risultati/v1_stato_fumo/log.jsonl', encoding='utf-8') as f:
    for r in [json.loads(r) for r in f]:
        print(r)

with open('dati/risultati/v1_stato_fumo/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))


### 13b. Run vero (v1_stato) — SOLO dopo l'ok esplicito di Andrea al fumo

Stima ~1-1,5h (6000 storie, 8000 step). Al termine: esame ufficiale di stadio 1
(confronto diretto con v1: 0,573) con decode interlacciato, e `esami.diagnosi`
su ENTRAMBI i checkpoint (l'ultimo e il best-dev) per lo split `esame`. La
sezione "stato" della diagnosi riporta l'accuratezza dei blocchi generati per
tick e per distanza dalla coda, più l'anatomia degli errori (tick vicino ±1-2 =
puntatore sfocato / luogo di un'altra persona = interferenza): la verifica
empirica diretta dell'ipotesi §0. Criteri di lettura fissati PRIMA del run in
fasi/FASE2_PIANO_STATO.md §9.


In [ ]:
# Copia di sicurezza dedicata al run vero.
import os
COPIA_DIR_STATO = os.path.join(CHECKPOINT_DIR, 'v1_stato')
os.makedirs(COPIA_DIR_STATO, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR_STATO)
parziale = os.path.join(COPIA_DIR_STATO, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')


In [ ]:
!python -m esami.genera --config configs/v1_stato.yaml --stadio 1


In [ ]:
stampa_composizione_stato('dati/stato/stadio1/train.jsonl')


In [ ]:
!python -m cervello.addestra --config configs/v1_stato.yaml --stadio 1 --copia-sicurezza "{COPIA_DIR_STATO}"


In [ ]:
import json

with open('dati/risultati/v1_stato/log.jsonl', encoding='utf-8') as f:
    for r in [json.loads(r) for r in f]:
        print(r)

with open('dati/risultati/v1_stato/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Niente copia manuale: checkpoint, log ed esame sono gia' stati replicati
# su Drive dal training (--copia-sicurezza).


In [ ]:
# Diagnosi (fasi/FASE2_PIANO_STATO.md §6.2/§9) su ENTRAMBI i checkpoint, split
# esame: la sezione "stato" e' attiva perche' configs/v1_stato.yaml ha
# dataset.stato. diagnosi.py non ha --copia-sicurezza: si copiano i JSON su
# Drive a mano (stessa logica della sezione 12).
import shutil, os

for checkpoint, suffisso in (('stadio1.pt', ''), ('stadio1_best.pt', '_best')):
    comando = (
        f'python -m esami.diagnosi --config configs/v1_stato.yaml --stadio 1 '
        f'--checkpoint dati/risultati/v1_stato/{checkpoint} --split esame'
    )
    get_ipython().system(comando)
    sorgente = 'dati/risultati/v1_stato/diagnosi_stadio1.json'
    dest = os.path.join(COPIA_DIR_STATO, f'diagnosi_stadio1{suffisso}.json')
    shutil.copy2(sorgente, dest)

print('diagnosi copiate su', COPIA_DIR_STATO)
